# Double Descent

## Classical Bias-Variance Recap

TODO: prose. The classical story: as model complexity increases, training error monotonically decreases but test error follows a U-shaped curve -- a sweet spot exists between underfitting and overfitting. This was the dominant mental model for generalization for decades. For a full treatment of the statistical learning foundations see [NB01](/notebooks/classical/01-bias-variance.html) in the Classical ML series.

In [ ]:
#| label: fig-bias-variance-classical
#| fig-cap: "Classical bias-variance tradeoff: train error (blue) and test error (red) as model complexity increases. The U-shaped test error curve suggests an optimal complexity. This picture breaks down in the modern overparameterized regime."
#| code-fold: true

# TODO: Reproduce classical bias-variance curve
# - Synthetic: polynomial regression on noisy sin data
# - Sweep polynomial degree 1..20
# - For each degree: 50 random train/test splits, compute mean train/test MSE
# - Plot mean ± std band for train error (blue) and test error (red)
# - Mark optimal degree with a dashed vertical line
# - X-axis: model complexity (degree), Y-axis: MSE

## Model-Wise Double Descent

TODO: prose. Belkin et al. (2019) observed that as model capacity continues to increase past the interpolation threshold (where the model can fit the training set perfectly), test error begins to decrease again -- a second descent. This produces a characteristic double-U shape. The interpolation threshold is the critical transition: just enough capacity to memorize the training data, but not enough to find a smooth solution.

In [ ]:
#| label: fig-model-wise-double-descent
#| fig-cap: "Model-wise double descent on CIFAR-10 with ResNets of increasing width. Test error rises to a peak at the interpolation threshold (where train error first reaches zero), then descends again as the model is further overparameterized. Reproduced from Nakkiran et al. (2019)."
#| code-fold: true

# TODO: Reproduce model-wise double descent (Nakkiran et al. Figure 1 / Figure 2)
# - Model: ResNet-18 with varying width multiplier k in {1, 2, 4, 8, 16, 32, 64}
# - Dataset: CIFAR-10 with label noise (15% random label corruption)
# - Train each model to convergence (or fixed epochs)
# - Record final train error and test error
# - Plot: X-axis = number of parameters (log scale), Y-axis = error (%)
# - Two curves: train error (blue), test error (red)
# - Mark interpolation threshold (train error -> 0) with dashed vertical
# - Label the three regimes: underparameterized, interpolation threshold, overparameterized
# - Note: this requires GPU; use checkpointed results if available

## Epoch-Wise Double Descent

TODO: prose. Nakkiran et al. also identified a temporal analog: for a fixed model near the interpolation threshold, test error can increase during training before decreasing again. This means early stopping can be harmful for models near this critical capacity, and the conventional wisdom of "stop when validation loss increases" may be wrong.

In [ ]:
#| label: fig-epoch-wise-double-descent
#| fig-cap: "Epoch-wise double descent for ResNets at three capacity levels: underparameterized (small $k$), near the interpolation threshold (critical $k$), and overparameterized (large $k$). The critical model shows a distinct bump in test error during training."
#| code-fold: true

# TODO: Reproduce epoch-wise double descent (Nakkiran et al. Figure 3)
# - Same ResNet/CIFAR-10 setup with label noise
# - Three models: small k (underparameterized), critical k (at interpolation threshold), large k
# - Record test error at every epoch (or every N epochs) during training
# - Three curves on same axes, one per model size
# - X-axis: epoch, Y-axis: test error (%)
# - Highlight the bump in the critical model curve with an annotation

## Reproducing Nakkiran et al. on ResNet / CIFAR-10

TODO: prose. Walk through the experimental setup in detail: ResNet-18 width multiplier, CIFAR-10 with label noise, optimizer settings, training duration. Discuss which aspects of the original paper are most important to match for the phenomenon to appear (label noise is critical -- without it the double descent is less pronounced).

![Nakkiran et al. (2019), Figure 8: phase diagram of test error as a function of model size (x-axis) and training epochs (y-axis). Dark regions = high error; the cross-shaped high-error region corresponds to the interpolation threshold in both dimensions.](img/nakkiran-phase-diagram.png){#fig-nakkiran-phase-diagram width=80%}

*Source: Nakkiran et al., "Deep Double Descent: Where Bigger Models and More Data Hurt", ICLR 2020. Reproduced for educational purposes.*

## Reconciliation with Modern Practice

TODO: prose. Why do overparameterized models generalize? Implicit regularization: gradient descent on overparameterized models converges to the minimum-norm solution (in linear models, this is provable). In neural networks, the loss landscape geometry and the inductive bias of SGD play analogous roles. Practical implications:

- **Early stopping** near the interpolation threshold can be harmful; consider training longer
- **Weight decay** shifts the interpolation threshold, interacting with model size in non-obvious ways
- **Model size** should not be constrained to the classical "sweet spot" -- larger is often better, especially with regularization
- **Label noise** exacerbates the double descent effect; cleaning data reduces it

The double descent phenomenon does not invalidate the bias-variance tradeoff -- it extends it. The U-curve holds within the underparameterized regime. Beyond the interpolation threshold, a different regime begins where overparameterization acts as an implicit regularizer.